# Email — Canonicalizing email addresses with Paxman

**Domain:** email  
**Capability contract:** `Email` (returns a frozen `CanonicalEmailContract`)  
**Public API:** `paxman.canonicalize(input_data, contract) -> ExecutionArtifact`

This notebook is the **reference template** for every capability notebook in this playground.
All notebooks share the same rhythm:

1. A title + scope header (this cell).
2. One imports cell — run top-to-bottom, no hidden state.
3. Concept markdown → runnable cells showing **different input spellings → canonical output**.
4. An error / edge-case cell near the end.
5. A pointer to the Engine and DSL notebooks for the bigger picture.

> Run every cell in order. `artifact.value` holds the canonical string when `artifact.status == "CANONICALIZED"`; otherwise it is `None`.

In [ ]:
from paxman import canonicalize, Email, ContractError, CanonicalizationError

def show(raw, contract):
    """Canonicalize `raw` and print status + canonical value (or why not)."""
    artifact = canonicalize(raw, contract)
    if artifact.status.name == "CANONICALIZED":
        print(f"{raw!r:35} -> {artifact.status.name:14} {artifact.value!r}")
    else:
        print(f"{raw!r:35} -> {artifact.status.name:14} (no canonical value)")
    return artifact

## The default contract

`Email()` lowercases and strips surrounding whitespace. The canonical form is the
mailbox `local@domain`. Different input spellings collapse to the same canonical output —
that is the point of canonicalization.

In [ ]:
c = Email()
show("A@B.COM", c)                 # upper -> lowercased
show("  a@b.com  ", c)             # surrounding whitespace stripped
show("a@b", c)                     # bare domain allowed; no dot required
# None / empty / whitespace-only are NOT guessed at a value:
show(None, c)                      # None -> UNSUPPORTED (no MISSING state for email)
show("", c)                        # ""   -> INVALID

## Provider aliases (Gmail normalization)

`provider_aliases="gmail"` applies Gmail's dot-stripping and `+tag` removal. This is a
**policy you declare on the contract** — Paxman never guesses it. Note what happens when
you *don't* declare Gmail aliasing on a dotted Gmail address: it becomes ambiguous.

In [ ]:
g = Email(provider_aliases="gmail")
show("John.Doe+spam@gmail.COM", g)   # dots + tag collapsed, lowercased
show("  John.Doe@Gmail.COM  ", g)     # same canonical result as above

n = Email(provider_aliases="none")
# Even with alias "none", a DOTTED Gmail address is AMBIGUOUS: Paxman will not
# guess whether the dot matters. It returns the candidate readings instead of
# a single canonical value (Identity: canonicalize only, never guess).
a = canonicalize("John.Doe@gmail.com", n)
print("status:", a.status.name)
print("candidates:", a.candidates)

## Errors & edge cases — Paxman refuses to guess

Malformed or ambiguous input is **not** partially parsed. It is reported as `INVALID`
(or `MISSING` for empty input), never silently corrected. A bad *contract* (not bad input)
raises `ContractError` at construction time.

In [ ]:
c = Email()
show("John Doe <a@b.com>", c)   # display-name form -> INVALID (not parsed)
show("foo@@bar.com", c)         # double @ -> INVALID

# A broken CONTRACT raises at construction, before canonicalize is ever called:
try:
    Email(provider_aliases="yahoo")   # not a valid alias value
except ContractError as exc:
    print("ContractError:", exc)

## Where to go next

- **`10_engine.ipynb`** — `Engine.default()`, `Engine.with_authorities(...)`, `canonicalize_with(...)`, and the `authority_override` escape hatch.
- **`11_dsl.ipynb`** — build contracts from a DSL string with `parse_contract`.
- All inputs above are sourced from `NOTEBOOK_INPUTS.md` (verified against the working tree).

> Every other capability notebook (boolean, country, date, geolocation, ip, money, phone, url, uuid) follows this exact structure.